In [3]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
df_temp = pd.read_csv("https://datahub.io/core/global-temp/r/0.csv")
df_temp.head()

,Source,Year,Mean
0,gcag,1850,-0.4177
1,gcag,1851,-0.2333
2,gcag,1852,-0.2294
3,gcag,1853,-0.2704
4,gcag,1854,-0.2915


In [11]:
chat = ChatOpenAI(model='gpt-3.5-turbo-0125')

agent_temp = create_pandas_dataframe_agent(
    chat,
    df_temp,
    verbose=True,
    agent_type="tool-calling",
    allow_dangerous_code=True  # ← adicione isso
)

In [12]:
resposta = agent_temp.invoke({"input": "Qual a temperatura média global do último ano?"})
print(resposta)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df[df['Year'] == df['Year'].max()]['Mean'].values[0]"}`


1.1755A temperatura média global do último ano registrada no DataFrame é de 1.1755.

> Finished chain.
{'input': 'Qual a temperatura média global do último ano?', 'output': 'A temperatura média global do último ano registrada no DataFrame é de 1.1755.'}


In [13]:
resposta = agent_temp.invoke({"input": "Qual foi o ano com a maior temperatura média?"})
print(resposta)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df[df['Mean'] == df['Mean'].max()]['Year']"}`


318    2024
Name: Year, dtype: int64O ano com a maior temperatura média registrada foi 2024.

> Finished chain.
{'input': 'Qual foi o ano com a maior temperatura média?', 'output': 'O ano com a maior temperatura média registrada foi 2024.'}


In [21]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_openai import ChatOpenAI
from pathlib import Path

# Constrói o caminho absoluto a partir do diretório do notebook
db_path = Path("files/sakila_master.db").resolve()

db_sakila = SQLDatabase.from_uri(f"sqlite:///{db_path}")

chat = ChatOpenAI(model='gpt-3.5-turbo-0125')

agent_executor_sakila = create_sql_agent(
    chat,
    verbose=True,
    db=db_sakila,
    agent_type="tool-calling",
)

In [22]:
resposta = agent_executor_sakila.invoke({"input": "Quais são os filmes mais alugados"})
print(resposta)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


actor, address, category, city, country, customer, film, film_actor, film_category, film_text, inventory, language, payment, rental, staff, store
Invoking: `sql_db_schema` with `{'table_names': 'film, rental'}`



CREATE TABLE film (
	film_id INTEGER NOT NULL, 
	title VARCHAR(255) NOT NULL, 
	description TEXT DEFAULT NULL, 
	release_year VARCHAR(4) DEFAULT NULL, 
	language_id SMALLINT NOT NULL, 
	original_language_id SMALLINT DEFAULT NULL, 
	rental_duration SMALLINT DEFAULT 3 NOT NULL, 
	rental_rate DECIMAL(4, 2) DEFAULT (4.99) NOT NULL, 
	length SMALLINT DEFAULT NULL, 
	replacement_cost DECIMAL(5, 2) DEFAULT (19.99) NOT NULL, 
	rating VARCHAR(10) DEFAULT 'G', 
	special_features VARCHAR(100) DEFAULT NULL, 
	last_update TIMESTAMP NOT NULL, 
	PRIMARY KEY (film_id), 
	CONSTRAINT fk_film_language FOREIGN KEY(language_id) REFERENCES language (language_id), 
	CONSTRAINT fk_film_language_original FOREIGN 

C:\Users\soste\AppData\Local\pypoetry\Cache\virtualenvs\curso-4-3-2-langchain-crie-agentes-ia-apps-OzPkRWV8-py3.12\Lib\site-packages\langchain_community\utilities\sql_database.py:363: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning may raise an error in a future release.
  metadata_table_names = [tbl.name for tbl in self._metadata.sorted_tables]
C:\Users\soste\AppData\Local\pypoetry\Cache\virtualenvs\curso-4-3-2-langchain-crie-agentes-ia-apps-OzPkRWV8-py3.12\Lib\site-packages\langchain_community\utilities\sql_database.py:375: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning 


Invoking: `sql_db_query` with `{'query': 'SELECT film.title, COUNT(rental.rental_id) as rental_count FROM film JOIN inventory ON film.film_id = inventory.film_id JOIN rental ON inventory.inventory_id = rental.inventory_id GROUP BY film.title ORDER BY rental_count DESC LIMIT 10'}`


[('BUCKET BROTHERHOOD', 34), ('ROCKETEER MOTHER', 33), ('SCALAWAG DUCK', 32), ('RIDGEMONT SUBMARINE', 32), ('JUGGLER HARDLY', 32), ('GRIT CLOCKWORK', 32), ('FORWARD TEMPLE', 32), ('ZORRO ARK', 31), ('WIFE TURN', 31), ('TIMBERLAND SKY', 31)]Os filmes mais alugados são:
1. BUCKET BROTHERHOOD - 34 aluguéis
2. ROCKETEER MOTHER - 33 aluguéis
3. SCALAWAG DUCK - 32 aluguéis
4. RIDGEMONT SUBMARINE - 32 aluguéis
5. JUGGLER HARDLY - 32 aluguéis
6. GRIT CLOCKWORK - 32 aluguéis
7. FORWARD TEMPLE - 32 aluguéis
8. ZORRO ARK - 31 aluguéis
9. WIFE TURN - 31 aluguéis
10. TIMBERLAND SKY - 31 aluguéis

> Finished chain.
{'input': 'Quais são os filmes mais alugados', 'output': 'Os filmes mais alugados são:\n1. BUCKET BROTHERHO